# Tutorial 4: General Utilities — Grid, NN, IDW, Kernel Smoothing

**Corresponds to MATLAB `GENLIBtutorial.m`**

This notebook demonstrates:
1. Creating estimation grids
2. Nearest-neighbour estimation
3. Inverse-distance-weighted (IDW) interpolation
4. Kernel smoothing
5. Comparing all three approaches

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from pybme import coord2dist

## Helper Functions

In [ ]:
def create_grid(origin, spacing, n_pts):
    """Create a regular estimation grid (like MATLAB creategrid)."""
    axes = [origin[i] + np.arange(n_pts[i]) * spacing[i]
            for i in range(len(origin))]
    grids = np.meshgrid(*axes, indexing='ij')
    return np.column_stack([g.ravel() for g in grids])


def inverse_distance(ck, ch, zh, power=2, nhmax=20, dmax=np.inf):
    """Inverse-distance-weighted interpolation. power=inf gives nearest-neighbour."""
    ck, ch = np.atleast_2d(ck), np.atleast_2d(ch)
    D = coord2dist(ck, ch)
    z_est = np.zeros(len(ck))
    for i in range(len(ck)):
        d = D[i]
        idx = np.where(d <= dmax)[0]
        if len(idx) == 0:
            z_est[i] = np.nan
            continue
        idx = idx[np.argsort(d[idx])][:nhmax]
        di = d[idx]
        if np.min(di) < 1e-10:
            z_est[i] = zh[idx[np.argmin(di)]]
        elif np.isinf(power):
            z_est[i] = zh[idx[0]]
        else:
            w = 1.0 / di ** power
            z_est[i] = np.sum(w * zh[idx]) / np.sum(w)
    return z_est


def kernel_smoothing(ck, ch, zh, bandwidth, nhmax=20, dmax=np.inf):
    """Gaussian kernel smoothing with bandwidth = σ²."""
    ck, ch = np.atleast_2d(ck), np.atleast_2d(ch)
    D = coord2dist(ck, ch)
    z_est = np.zeros(len(ck))
    for i in range(len(ck)):
        d = D[i]
        idx = np.where(d <= dmax)[0]
        if len(idx) == 0:
            z_est[i] = np.nan
            continue
        idx = idx[np.argsort(d[idx])][:nhmax]
        w = np.exp(-0.5 * d[idx] ** 2 / bandwidth)
        z_est[i] = np.sum(w * zh[idx]) / np.sum(w)
    return z_est

## 1. Generate Synthetic Data & Grid

In [ ]:
rng = np.random.default_rng(42)

def true_field(x, y):
    return 50 + 15 * np.sin(x / 3) + 10 * np.cos(y / 2.5)

n = 30
ch = np.column_stack([rng.uniform(0, 20, n), rng.uniform(0, 20, n)])
zh = true_field(ch[:, 0], ch[:, 1]) + rng.normal(0, 3, n)

grid = create_grid(origin=[0, 0], spacing=[1, 1], n_pts=[21, 21])
z_true = true_field(grid[:, 0], grid[:, 1])

print(f"Data points: {n}")
print(f"Grid: 21×21 = {len(grid)} points")

## 2. Run All Three Methods

In [ ]:
# Nearest-neighbour
z_nn = inverse_distance(grid, ch, zh, power=np.inf, nhmax=20, dmax=50)
rmse_nn = np.sqrt(np.nanmean((z_nn - z_true) ** 2))

# IDW power=2
z_idw = inverse_distance(grid, ch, zh, power=2, nhmax=20, dmax=50)
rmse_idw = np.sqrt(np.nanmean((z_idw - z_true) ** 2))

# Kernel smoothing
z_ks = kernel_smoothing(grid, ch, zh, bandwidth=5.0, nhmax=20, dmax=50)
rmse_ks = np.sqrt(np.nanmean((z_ks - z_true) ** 2))

print(f"{'Method':20s}  {'RMSE':>8s}")
print("-" * 32)
print(f"{'Nearest-neighbour':20s}  {rmse_nn:8.3f}")
print(f"{'IDW (power=2)':20s}  {rmse_idw:8.3f}")
print(f"{'Kernel smoothing':20s}  {rmse_ks:8.3f}")

## 3. Visual Comparison

In [ ]:
gx = grid[:, 0].reshape(21, 21)
gy = grid[:, 1].reshape(21, 21)

fig, axes = plt.subplots(2, 2, figsize=(11, 10))
titles = ['True Field', 'Nearest Neighbour', 'IDW (power=2)', 'Kernel Smoothing']
data = [z_true, z_nn, z_idw, z_ks]

for ax, Z, title in zip(axes.ravel(), data, titles):
    im = ax.pcolormesh(gx, gy, Z.reshape(21, 21), cmap='hot', shading='auto')
    ax.scatter(ch[:, 0], ch[:, 1], c='cyan', marker='v', s=15, zorder=5)
    ax.set_title(title)
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax)

fig.suptitle('Estimation Methods Comparison', fontsize=13)
fig.tight_layout()
plt.show()